In [9]:
import pandas as pd

from Bio import SeqIO
from modules.sequences import (
    split_train_val_test,
    remove_unknown_amino_acids
)

from modules.model import (
    init_transition_mat,
    remove_silent_states,
    init_emission_mat
)

from modules.train import train_phmm
from modules.evaluation import description_accuracy
from modules.model import build_hmmlearn_model

from pathlib import Path
import logging
import joblib

import numpy as np

In [12]:

family_df = pd.read_csv("protein_families.csv")

max_iter = 15
tol = 1e-3
ts = 1.0
es = 5.0

# Directory for saved models and histories
model_dir = Path("saved_models/phmm/main")
model_dir.mkdir(parents=True, exist_ok=True)

# Consolidated results across protein families
summary = []

for i in range(len(family_df)):
    name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]
    label = f"{name}_{accession}"

    print(f"\n--- Training {label} ---", flush=True)

    # ==========================================
    # 1. LOAD AND SPLIT SEQUENCES
    # ==========================================

    recs = [
        (r.id, str(r.seq))
        for r in SeqIO.parse(
            f"sequences/{accession}.fasta", "fasta"
        )
    ]

    train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

    remove_train, train_seqs = remove_unknown_amino_acids(train_seqs)
    remove_val, val_seqs = remove_unknown_amino_acids(val_seqs)
    remove_test, test_seqs = remove_unknown_amino_acids(test_seqs)

    print(f"Total removed sequences in {name}-{accession}")
    print(f"Train set: {remove_train}")
    print(f"Validation set: {remove_val}")
    print(f"Test set: {remove_test}")

    # ==========================================
    # 2. INITIALIZE PHMM
    # ==========================================

    trans_matdf = init_transition_mat(match_len)
    emit_matdf = init_emission_mat(match_len, seed=2)

    # Remove silent D states
    cols = list(trans_matdf.columns)
    d_cols = [col for col in cols if col[0] == "D"]

    trans_mat, red_cols = remove_silent_states(
        trans_matdf.to_numpy(),
        cols,
        d_cols
    )

    trans_matdf = pd.DataFrame(
        trans_mat,
        columns=red_cols,
        index=red_cols
    )

    A = trans_mat.T
    B = emit_matdf.to_numpy().T

    startprob = np.zeros(A.shape[0])
    startprob[0] = 1.0

    # Encode symbols
    symbols = list(emit_matdf.index)
    sym_idx = {
        symbol: j
        for j, symbol in enumerate(symbols)
    }

    BEGIN = sym_idx["^"]
    END = sym_idx["$"]

    # ==========================================
    # 3. EVALUATE UNTRAINED MODEL
    # ==========================================

    initial_model = build_hmmlearn_model(
        A, B, startprob
    )

    da0 = description_accuracy(
        initial_model,
        sym_idx,
        test_seqs
    )

    initial_mean = float(da0.mean())
    initial_std = float(da0.std())

    print(
        f"Initial Test DA: "
        f"{initial_mean:.4f} ± {initial_std:.4f}",
        flush=True
    )

    # ==========================================
    # 4. TRAIN PHMM
    # ==========================================

    model, hist = train_phmm(
        build_hmmlearn_model(
            A,
            B,
            startprob,
            trans_strength=ts,
            emit_strength=es
        ),
        sym_idx,
        train_seqs,
        val_seqs,
        test_seqs,
        max_iter,
        tol
    )

    # ==========================================
    # 5. EVALUATE TRAINED MODEL
    # ==========================================

    da = description_accuracy(
        model,
        sym_idx,
        test_seqs
    )

    trained_mean = float(da.mean())
    trained_std = float(da.std())

    print(
        f"Trained Test DA: "
        f"{trained_mean:.4f} ± {trained_std:.4f}",
        flush=True
    )

    # ==========================================
    # 6. SAVE MODEL AND TRAINING HISTORY
    # ==========================================

    prefix = f"{name}_{accession}_m{match_len}"

    model_path = model_dir / f"{prefix}.joblib"
    history_path = model_dir / f"{prefix}_history.csv"

    # Save trained model
    joblib.dump(model, model_path)

    # Save training history
    hist.to_csv(history_path, index=False)

    print(f"Model saved: {model_path}", flush=True)
    print(f"History saved: {history_path}", flush=True)

    # ==========================================
    # 7. COLLECT SUMMARY RESULTS
    # ==========================================

    summary.append({
        "name": name,
        "accession": accession,
        "m": match_len,
        "n": A.shape[0],
        "train_size": len(train_seqs),
        "val_size": len(val_seqs),
        "test_size": len(test_seqs),
        "initial_test_da_mean": initial_mean,
        "initial_test_da_std": initial_std,
        "trained_test_da_mean": trained_mean,
        "trained_test_da_std": trained_std,
        "max_iter": max_iter,
        "tol": tol,
        "trans_strength": ts,
        "emit_strength": es,
    })

    # Save summary after every protein family
    # to retain completed results if interrupted.
    summary_df = pd.DataFrame(summary)

    summary_path = model_dir / "phmm_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print(f"Summary updated: {summary_path}", flush=True)

# ==========================================
# 8. FINAL SUMMARY
# ==========================================

summary_df = pd.DataFrame(summary)

display(summary_df)

print("\nPHMM training completed.")



--- Training DUF7563_PF24444 ---
Total removed sequences in DUF7563-PF24444
Train set: 0
Validation set: 0
Test set: 0
Initial Test DA: 0.0013 ± 0.0026


Fitting a model with 7899 free scalar parameters with only 4887 data points will result in a degenerate solution.


iter   0  cpu     0.0s  log-lik -14656.90  train_da_mean 0.0023  val_da_mean 0.0033


Fitting a model with 7899 free scalar parameters with only 4887 data points will result in a degenerate solution.


iter   1  cpu     0.1s  log-lik -13179.05  train_da_mean 0.1080  val_da_mean 0.1024


Fitting a model with 7899 free scalar parameters with only 4887 data points will result in a degenerate solution.


... (77 outputs left out)

Fitting a model with 7899 free scalar parameters with only 7678 data points will result in a degenerate solution.


iter  14  cpu     2.6s  log-lik -17085.00  train_da_mean 0.3371  val_da_mean 0.3198
iter  15  cpu     2.8s  log-lik -17080.73  train_da_mean 0.3374  val_da_mean 0.3200
Trained Test DA: 0.3182 ± 0.0897
Model saved: saved_models/phmm/main/Nuo9p5_PF30687_m39.joblib
History saved: saved_models/phmm/main/Nuo9p5_PF30687_m39_history.csv
Summary updated: saved_models/phmm/main/phmm_summary.csv

--- Training Ribosomal_uL15_N_PF29471 ---
Total removed sequences in Ribosomal_uL15_N-PF29471
Train set: 0
Validation set: 0
Test set: 0
Initial Test DA: 0.0065 ± 0.0026
iter   0  cpu     0.0s  log-lik -39202.98  train_da_mean 0.0063  val_da_mean 0.0060
iter   1  cpu     0.2s  log-lik -35746.25  train_da_mean 0.1012  val_da_mean 0.1169
iter   2  cpu     0.5s  log-lik -34779.80  train_da_mean 0.1280  val_da_mean 0.1448
iter   3  cpu     0.8s  log-lik -34223.57  train_da_mean 0.1435  val_da_mean 0.1572
iter   4  cpu     1.0s  log-lik -33843.02  train_da_mean 0.1541  val_da_mean 0.1672
iter   5  cpu     1.

,name,accession,m,n,train_size,val_size,test_size,initial_test_da_mean,initial_test_da_std,trained_test_da_mean,trained_test_da_std,max_iter,tol,trans_strength,emit_strength
0,DUF7563,PF24444,39,79,70,10,20,0.001251,0.002582,0.340656,0.072592,15,0.001,1.0,5.0
1,JmjN,PF02375,34,69,70,10,20,0.008281,0.002490,0.054818,0.019405,15,0.001,1.0,5.0
2,KOW,PF00467,32,65,70,10,20,0.006485,0.001956,0.072097,0.018239,15,0.001,1.0,5.0
3,MerR,PF00376,38,77,69,10,20,0.005869,0.002268,0.135491,0.036137,15,0.001,1.0,5.0
4,SapB_2,PF03489,34,69,70,10,20,0.008414,0.001273,0.060030,0.025315,15,0.001,1.0,5.0
5,DnaJ_C_2,PF27439,34,69,70,10,20,0.008709,0.000700,0.103022,0.007230,15,0.001,1.0,5.0
6,DUF1232,PF06803,37,75,70,10,20,0.005565,0.002877,0.062218,0.023116,15,0.001,1.0,5.0
7,DUF1328,PF07043,39,79,70,10,20,0.001410,0.003250,0.412816,0.076113,15,0.001,1.0,5.0
8,DUF5808,PF19124,26,53,70,10,20,0.007185,0.002702,0.084915,0.029159,15,0.001,1.0,5.0
9,E3_binding,PF02817,36,73,70,10,20,0.008536,0.000440,0.110127,0.021879,15,0.001,1.0,5.0



PHMM training completed.


In [14]:
emit_matdf.index

Index(['^', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P',
       'Q', 'R', 'S', 'T', 'V', 'W', 'Y', '$'],
      dtype='str')